# Experiment 1 · Box-conditioned traditional baselines

The three traditional baselines — **U-Net** (ResNet50 encoder), **TransUNet** (R50-ViT-B/16), and **nnU-Net v2** — but now each receives the oracle bounding box as an **extra binary input channel** (channel-concatenation, following MedSAM / Ma et al. 2024). The box is rasterised as a 4th `{0,1}` channel so the CNN gets the same spatial prior the box-prompted foundation models get. During **training** the box is jittered by **±10%** (seeded, epoch-varying); at **evaluation** the **tight** box (no jitter) is used, matching the few-shot LoRA eval so Figure 1 is apples-to-apples. The 4th-channel conv is zero-initialised, so at init the box-CNN is identical to its image-only twin. Everything runs with the fixed global **seed 42** and patient-level splits. This produces the "box-conditioned" block of paper Table 1.

In [ ]:
import os
from pathlib import Path
while not (Path.cwd() / 'pyproject.toml').exists() and Path.cwd() != Path.cwd().parent:
    os.chdir('..')
print('repo root:', Path.cwd())

In [ ]:
# Check the inputs this experiment needs before doing anything slow. A missing
# dataset here means an unrun (or unplaced) setup step, not a bug in the experiment.
from pathlib import Path

REQUIRED = ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']
GATED = {'thyroidxl', 'stanford_aimi'}

missing = [d for d in REQUIRED
           if not (Path('data/processed') / d / 'images').is_dir()
           or not (Path('data/splits') / f'{d}_test.csv').exists()]
if missing:
    print('Missing preprocessed data or splits for:', ', '.join(missing))
    for d in missing:
        if d in GATED:
            print(f'  {d:14s} access-gated -> place your approved copy first, see '
                  f'00_setup/01_place_gated_datasets.ipynb')
        else:
            print(f'  {d:14s} open -> download it with 00_setup/00_get_open_datasets.ipynb')
    print('Then run 00_setup/02_preprocess.ipynb and 00_setup/03_make_splits.ipynb.')
    print('\nYou can still run this experiment on whichever datasets ARE present '
          'by restricting the --dataset argument below.')
else:
    print('All four datasets are preprocessed and split.')


## Run

The experiment is a 3 models × 4 datasets sweep. U-Net and TransUNet share one entrypoint (`run.py`, selected with `--model`); each does its own per-dataset LR search internally. nnU-Net v2 has its own framework and is driven by a separate shell pipeline (below).

In [ ]:
# U-Net and TransUNet, box-conditioned: one run per (model, dataset). Results -> results/
for model in ['unet', 'transunet']:
    for dataset in ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']:
        !python experiments/exp1_fullsupervised/traditional_boxcond/run.py --model {model} --dataset {dataset}

### nnU-Net v2 (box-conditioned)

nnU-Net is self-configuring and needs its own dataset conversion + plan/preprocess/train/predict pipeline. Box-conditioned datasets use IDs `011`–`014` (`011`=DDTI, `012`=TN3K, `013`=ThyroidXL, `014`=Stanford AIMI) to avoid clashing with the image-only `001`–`004`. The image goes in channel 0 (z-scored) and the tight oracle box in channel 1 (`nonorm`, kept `{0,1}`). Convert with `convert_to_nnunet_boxcond.py`, run the pipeline per dataset number, then evaluate with `eval_nnunet_boxcond.py` (or run `run_nnunet_boxcond_pipeline.sh` to do all four end-to-end).

In [ ]:
# !bash experiments/exp1_fullsupervised/traditional_boxcond/run_nnunet_boxcond_pipeline.sh 011   # (long; self-configured nnU-Net, 1000 epochs)

## Results

The aggregate lives in `results/exp1_boxcond_summary.csv` — one row per
(model, dataset) with test DSC / IoU / precision / recall / HD95 for all three
box-conditioned baselines (U-Net, TransUNet, nnU-Net v2). It is rebuilt from the
per-model fragments by `aggregate_boxcond.py` and shown below.

In [ ]:
import pandas as pd
df = pd.read_csv('experiments/exp1_fullsupervised/traditional_boxcond/results/exp1_boxcond_summary.csv')
df